In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [25]:
df=pd.read_csv('../data/churn_clean.csv')

In [26]:
for col in ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'Churn']:
    df[col] = df[col].map({'Yes': 1, 'No': 0})
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})


plusieurs categorie sans ordre

One-Hot Encoding
**Au lieu d'attribuer un numéro à chaque catégorie, on crée une nouvelle colonne pour chaque catégorie.**

In [27]:
df=pd.get_dummies(df, columns=['Contract', 'PaymentMethod', 'InternetService'],dtype=int, drop_first=True)

In [28]:
x=df.drop('Churn', axis=1)
y=df['Churn']

In [29]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [30]:
##verification 
print(df.isna().sum().sum())
print(x_train.dtypes[x_train.dtypes == "object"])

0
Series([], dtype: object)


Ce que fait StandardScaler

Il remet toutes les colonnes numériques sur une échelle comparable, avec cette formule pour chaque valeur :

nouvelle valeur = (valeur - moyenne de la colonne) / écart-type de la colonne

Après ça, chaque colonne a une moyenne de 0 et un écart-type de 1. Une valeur à 0 veut dire « dans la moyenne », +1 « un peu au-dessus », -2 « très en dessous ».

In [31]:
x_train.describe()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaperlessBilling,MonthlyCharges,TotalCharges,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_Fiber optic,InternetService_No
count,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,...,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000,5634.000000
mean,0.502840,0.163294,0.484381,0.298012,32.485091,0.900781,0.424210,0.288072,0.351083,0.345758,...,0.591232,64.929961,2299.334682,0.208200,0.241214,0.215300,0.335641,0.228257,0.440717,0.215477
std,0.500036,0.369667,0.499800,0.457425,24.568744,0.298982,0.494266,0.452905,0.477351,0.475657,...,0.491650,30.138105,2279.204278,0.406057,0.427858,0.411067,0.472256,0.419746,0.496517,0.411190
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,18.400000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,35.662500,402.975000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,29.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,70.500000,1394.925000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,0.000000,1.000000,1.000000,55.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,90.000000,3835.825000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,72.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,118.750000,8684.800000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


fit_transform() fait deux opérations :
- fit() : calcule la moyenne et l'écart-type de chaque colonne à partir de X_train.
- transform() : transforme les valeurs en utilisant ces paramètres.

In [32]:
scaler = StandardScaler()
x_train[['tenure','MonthlyCharges','TotalCharges']] = scaler.fit_transform(x_train[['tenure','MonthlyCharges','TotalCharges']])   
x_test[['tenure','MonthlyCharges','TotalCharges']]  = scaler.transform(x_test[['tenure','MonthlyCharges','TotalCharges']])      

In [33]:
print(scaler.mean_)
print(scaler.scale_)
print(x_train[['tenure', 'MonthlyCharges', 'TotalCharges']].describe())

[  32.48509052   64.92996095 2299.33468229]
[  24.56656302   30.13543058 2279.0019965 ]
             tenure  MonthlyCharges  TotalCharges
count  5.634000e+03    5.634000e+03  5.634000e+03
mean  -1.008935e-17   -2.402527e-16  2.522338e-17
std    1.000089e+00    1.000089e+00  1.000089e+00
min   -1.322329e+00   -1.544028e+00 -1.008922e+00
25%   -9.559779e-01   -9.711977e-01 -8.321009e-01
50%   -1.418632e-01    1.848336e-01 -3.968446e-01
75%    9.164859e-01    8.319124e-01  6.741944e-01
max    1.608483e+00    1.785939e+00  2.801869e+00


In [34]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
bin_cols = [c for c in x_train.columns if c not in num_cols]

print("1. NaN:", df.isna().sum().sum(), x_train.isna().sum().sum(), x_test.isna().sum().sum())
print("2. Colonnes texte:", list(x_train.select_dtypes(include="object").columns))
print("3. customerID présent:", "customerID" in x_train.columns)
print("4. Shapes:", x_train.shape, x_test.shape, y_train.shape, y_test.shape)
print("5. Churn train/test:", round(y_train.mean(), 4), round(y_test.mean(), 4))
print("6. Indices en commun:", len(x_train.index.intersection(x_test.index)))
print("7. Binaires dans {0,1}:", x_train[bin_cols].isin([0, 1]).all().all())
print("8. Scaler:", scaler.n_features_in_, scaler.mean_.round(1))
print("9. Train scalé:")
print(x_train[num_cols].agg(["mean", "std"]).round(3))
print("10. Test scalé:")
print(x_test[num_cols].agg(["mean", "std"]).round(3))
orig = pd.DataFrame(scaler.inverse_transform(x_test[num_cols]), columns=num_cols)
print("11. Test remis à l'échelle d'origine:")
print(orig.agg(["min", "max"]).round(1))
print("12. Colonnes:", len(x_train.columns))
print(list(x_train.columns))

1. NaN: 0 0 0
2. Colonnes texte: []
3. customerID présent: False
4. Shapes: (5634, 23) (1409, 23) (5634,) (1409,)
5. Churn train/test: 0.2654 0.2654
6. Indices en commun: 0
7. Binaires dans {0,1}: True
8. Scaler: 3 [  32.5   64.9 2299.3]
9. Train scalé:
      tenure  MonthlyCharges  TotalCharges
mean    -0.0            -0.0           0.0
std      1.0             1.0           1.0
10. Test scalé:
      tenure  MonthlyCharges  TotalCharges
mean  -0.023          -0.028        -0.043
std    0.998           0.992         0.972
11. Test remis à l'échelle d'origine:
     tenure  MonthlyCharges  TotalCharges
min     0.0            18.2           0.0
max    72.0           116.8        8496.7
12. Colonnes: 23
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Contract_One year', 'Contract_Two yea

In [35]:
# Notebook : on enregistre
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(list(x_train.columns), "../models/columns.pkl")


['../models/columns.pkl']